# Baseline classifier: training, testing, running

Fine-tunes an Ultralytics YOLO classification model (`yolo26n-cls.pt`) on
SID-Set to tell real photos from AI-generated/tampered ones.

This notebook drives the pipeline entirely through the shared classes in
`packages/models/normal_classifier`, instead of ad-hoc inline code:

- **Training + testing** — `NormalClassifierTrainer`, which extends
  `shared_types.TrainableModel` (`.train()` / `.evaluate()` / `.save()` / `.load()`).
- **Running (inference)** — `NormalClassifierDetector`, which implements
  `shared_types.EnsembleDetector` (`.predict()`) — the same "ready" contract
  `apps/web`'s Streamlit demo already knows how to consume.

Data comes from `data.dataset_builder`, which streams SID-Set from Hugging
Face rather than downloading it, and never holds more than one decoded
image in memory at a time:

- `iter_sid_subset()` — a one-shot stream for the large, single-use
  training pull.
- `sid_subset_factory()` — for the small validation set, which is iterated
  three times below (train's val pass, `evaluate()`, the inference demo).
  It returns a callable that hands back a *fresh* one-image-at-a-time
  stream on each call, so the set is never materialized in RAM; the pull
  is seed-deterministic, so every pass sees the same images.

Both yield the shared `LabeledImageSample` type the classes above expect.
(`load_sid_subset()` + `to_labeled_samples()` still exist for the
materialized case, but this notebook avoids them to keep Colab RAM flat.)


## 1. Setup — clone the repo, install deps, wire up imports

In [ ]:
%cd /content/
!git clone https://github.com/Zhongbob/TikTokTechJam2026.git

In [ ]:
%cd /content/TikTokTechJam2026
!git switch "setup"
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os,sys
os.environ["PATH"] += f":{os.path.expanduser('~/.local/bin')}"
os.environ["UV_PROJECT_ENVIRONMENT"] = sys.prefix
%cd /content/TikTokTechJam2026/packages/models/normal_classifier
!uv sync


In [ ]:
!uv pip install --system ../../data
# Might need to install local packages manually if the above doesnt work

# RESTART SESSION UNDER RUNTIME AFTER COMPLETING THE ABOVE

In [ ]:
# Run this AFTER restarting the session (cell above). Locates the .venv
# `uv sync` created and bridges its site-packages into this fresh kernel
# process via site.addsitedir() -- the API that actually processes uv's
# editable-install ".pth" redirects; a plain sys.path.insert() does not.
import glob
import importlib
import site
from pathlib import Path

candidates = [
    Path("/content/TikTokTechJam2026/.venv"),
    Path("/content/TikTokTechJam2026/packages/models/normal_classifier/.venv"),
]
venv_dir = next((c for c in candidates if c.is_dir()), None)
if venv_dir is None:
    found = glob.glob("/content/TikTokTechJam2026/**/.venv", recursive=True)
    assert found, "no .venv found under the repo -- did `uv sync` in the cell above actually succeed?"
    venv_dir = Path(found[0])

site_packages = next(venv_dir.glob("lib/python*/site-packages"))
print(f"bridging {site_packages}")
site.addsitedir(str(site_packages))

importlib.invalidate_caches()
importlib.import_module("normal_classifier")
print("'normal_classifier' imported OK")


## 2. Load training + validation data from SID-Set

In [ ]:
from data.dataset_builder import iter_sid_subset, sid_subset_factory

# Both pulls are lazy streams -- nothing holds more than one decoded image
# in memory at a time, so RAM stays flat regardless of how many images we
# pull.

# Training pull is large (1000/label) and consumed exactly once, by
# trainer.train() below, which writes each image to disk as it arrives.
train_samples = iter_sid_subset(images_per_label=1000, split="train")

# Validation pull is small but iterated three times (train's val pass,
# evaluate(), the inference demo). A generator is single-use, so instead of
# materializing it with load_sid_subset() + to_labeled_samples() -- which
# would pin every val image in RAM for the whole session -- we keep a
# factory and call it for a fresh stream each time. The pull is
# seed-deterministic, so every pass sees the same images in the same order.
make_val_samples = sid_subset_factory(images_per_label=200, split="validation")

print("train_samples + make_val_samples() are lazy streams -- consumed one image at a time downstream")


## 3. Train

`NormalClassifierTrainer.train()` exports `train_samples`/`val_samples` into
the `real/`, `ai_generated/` class-folder layout Ultralytics' classification
trainer expects, then fine-tunes `yolo26n-cls.pt` on them.

In [ ]:
from normal_classifier import NormalClassifierTrainer

trainer = NormalClassifierTrainer(base_weights="yolo26n-cls.pt", image_size=224)
result = trainer.train(
    train_samples,
    val_samples=make_val_samples(),
    output_dir="SID_YOLO",
    epochs=100,
    batch=32,
    patience=10,
    device="cpu",
    plots=True,
)
result


## 4. Test

The "testing" stage — score the trained model against a held-out set via
`.evaluate()`. SID-Set only exposes train/validation splits, so this
re-streams the same validation set (`make_val_samples()` again); swap in a
separate held-out set here if you have one.


In [ ]:
metrics = trainer.evaluate(make_val_samples(), output_dir="SID_YOLO_eval")
print("Held-out evaluation metrics:", metrics)


In [ ]:
trainer.save("normal_classifier.pt")
print("Saved checkpoint to normal_classifier.pt")


## 5. Run (inference)

The "running" stage — `NormalClassifierDetector` wraps the saved checkpoint
and implements the same `EnsembleDetector` contract `apps/web` consumes, so
this class can be dropped straight into
`apps/web/src/web/services/factory.py`'s `get_detector()` once ready.

In [ ]:
import itertools

from normal_classifier import NormalClassifierDetector

detector = NormalClassifierDetector.from_checkpoint("normal_classifier.pt")

# make_val_samples() again -- a fresh stream, only the 5 images we pull are
# ever decoded.
for sample in itertools.islice(make_val_samples(), 5):
    detection = detector.predict(sample.image)
    print(
        f"true={sample.metadata['label_name']:<10} "
        f"predicted={detection.verdict:<12} "
        f"p(ai_generated)={detection.ai_generated_probability:.2f}"
    )
